In [2]:
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================
# USER SETTINGS
# ============================================================
CSV_FILE = "/mnt/lustre/proj/kimyy/Observation/GLODAP/GLODAPv2.2023_all/GLODAPv2.2023_Pacific_Ocean.csv"
OUT_NC   = "/mnt/lustre/proj/kimyy/tr_sysong/OBS/GLODAP/NWP_GLODAP_v2_3_1deg_monthly.nc"

QC_MAX = 2

lat_range = (19, 41)
lon_range = (120, 180)

# model z_t in meters (CESM POP midpoints 예시)
z_t_model = np.array([
     5, 15, 25, 35, 45, 55, 65, 75, 85, 95,
    110, 130, 160, 200, 250, 310, 380, 460, 550, 650,
    760, 880, 1010, 1150, 1300
])

# ============================================================
# 1) Read CSV
# ============================================================
df = pd.read_csv(CSV_FILE, low_memory=False)
df = df.replace(-9999, np.nan)

# 컬럼명 공백 제거 (안전)
df.columns = df.columns.str.strip()

# ============================================================
# 2) Time handling (G2year / G2month)
# ============================================================
df["year"]  = df["G2year"].astype(int)
df["month"] = df["G2month"].astype(int)
df["ym"]    = df["year"] * 100 + df["month"]

# restrict period
df = df[(df["year"] >= 1955) & (df["year"] <= 2020)]

# ============================================================
# 3) Spatial + depth filtering
# ============================================================
df = df[
    (df["G2latitude"]  >= lat_range[0]) &
    (df["G2latitude"]  <= lat_range[1]) &
    (df["G2longitude"] >= lon_range[0]) &
    (df["G2longitude"] <= lon_range[1])
]

# depth in meters (positive downward)
df["depth_m"] = df["G2depth"]

# ============================================================
# 4) QC filtering (G2xxxqc)
# ============================================================
VAR_INFO = {
    "G2temperature": None,
    "G2salinity":    "G2salinityqc",
    "G2sigma0":      None,
    "G2oxygen":      "G2oxygenqc",
    "G2nitrate":     "G2nitrateqc",
    "G2phosphate":   "G2phosphateqc",
    "G2silicate":    "G2silicateqc",
    "G2tco2":        "G2tco2qc",
    "G2talk":        "G2talkqc",
}

for v, qc in VAR_INFO.items():
    if qc is not None and qc in df.columns:
        df.loc[df[qc] > QC_MAX, v] = np.nan

# ============================================================
# 5) 1×1 degree binning
# ============================================================
df["lat"] = np.floor(df["G2latitude"]).astype(int)
df["lon"] = np.floor(df["G2longitude"]).astype(int)

# ============================================================
# 6) Assign model z_t bins
# ============================================================
z_edges = np.concatenate((
    [0],
    (z_t_model[:-1] + z_t_model[1:]) / 2,
    [z_t_model[-1] + 500]
))

df["z_t"] = pd.cut(
    df["depth_m"],
    bins=z_edges,
    labels=z_t_model,
    include_lowest=True
).astype(float)

# drop obs not mapped to any z_t
df = df.dropna(subset=["z_t"])

# ============================================================
# 7) Monthly × 1deg × z_t averaging
# ============================================================
vars_use = [
    "G2temperature", "G2salinity", "G2sigma0",
    "G2oxygen", "G2nitrate", "G2phosphate",
    "G2silicate", "G2tco2", "G2talk"
]

df_mon = (
    df
    .groupby(["ym", "lat", "lon", "z_t"], observed=True)[vars_use]
    .mean()
    .reset_index()
)

# ============================================================
# 8) Convert to xarray & reindex (NaNs preserved)
# ============================================================
lat_new = np.arange(lat_range[0], lat_range[1] + 1)
lon_new = np.arange(lon_range[0], lon_range[1] + 1)

ds = (
    df_mon
    .set_index(["ym", "z_t", "lat", "lon"])
    .to_xarray()
    .reindex(
        lat=lat_new,
        lon=lon_new,
        z_t=z_t_model
    )
)

# ============================================================
# 9) Save
# ============================================================
ds.to_netcdf(OUT_NC)

print(ds)
print("Saved to:", OUT_NC)


<xarray.Dataset> Size: 707MB
Dimensions:        (ym: 280, z_t: 25, lat: 23, lon: 61)
Coordinates:
  * ym             (ym) int64 2kB 197310 197311 198505 ... 202009 202010 202011
  * z_t            (z_t) float64 200B 5.0 15.0 25.0 ... 1.15e+03 1.3e+03
  * lat            (lat) int64 184B 19 20 21 22 23 24 25 ... 36 37 38 39 40 41
  * lon            (lon) int64 488B 120 121 122 123 124 ... 176 177 178 179 180
Data variables:
    G2temperature  (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2salinity     (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2sigma0       (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2oxygen       (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2nitrate      (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2phosphate    (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2silicate     (ym, z_t, lat, lon) float64 79MB nan nan nan ... nan nan nan
    G2tco2    